# IA não supervisionada: segmentação de clientes

Este notebook mostra um exemplo simples de IA não supervisionada usando `scikit-learn`.

A ideia é encontrar grupos de clientes parecidos, sem uma coluna pronta dizendo qual é o segmento de cada um.

## Problema

Fato: a base não tem rótulo de segmento.

Inferência: depois que o K-Means encontra os grupos, nós interpretamos os centroides e damos nomes aos perfis.

Opinião técnica: K-Means é bom para este exemplo porque é direto, visual e muito usado para explicar agrupamento.

In [1]:
import os

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_columns', None)

In [2]:
def criar_base_clientes() -> pd.DataFrame:
    dados_clientes = [
        {'cliente': 'C01', 'compras_mes': 2, 'gasto_medio_reais': 35, 'dias_desde_ultima_compra': 90},
        {'cliente': 'C02', 'compras_mes': 1, 'gasto_medio_reais': 20, 'dias_desde_ultima_compra': 120},
        {'cliente': 'C03', 'compras_mes': 3, 'gasto_medio_reais': 45, 'dias_desde_ultima_compra': 80},
        {'cliente': 'C04', 'compras_mes': 8, 'gasto_medio_reais': 180, 'dias_desde_ultima_compra': 20},
        {'cliente': 'C05', 'compras_mes': 10, 'gasto_medio_reais': 220, 'dias_desde_ultima_compra': 14},
        {'cliente': 'C06', 'compras_mes': 7, 'gasto_medio_reais': 150, 'dias_desde_ultima_compra': 25},
        {'cliente': 'C07', 'compras_mes': 4, 'gasto_medio_reais': 80, 'dias_desde_ultima_compra': 45},
        {'cliente': 'C08', 'compras_mes': 5, 'gasto_medio_reais': 95, 'dias_desde_ultima_compra': 38},
        {'cliente': 'C09', 'compras_mes': 6, 'gasto_medio_reais': 110, 'dias_desde_ultima_compra': 35},
        {'cliente': 'C10', 'compras_mes': 12, 'gasto_medio_reais': 260, 'dias_desde_ultima_compra': 7},
        {'cliente': 'C11', 'compras_mes': 9, 'gasto_medio_reais': 210, 'dias_desde_ultima_compra': 12},
        {'cliente': 'C12', 'compras_mes': 2, 'gasto_medio_reais': 60, 'dias_desde_ultima_compra': 65},
        {'cliente': 'C13', 'compras_mes': 4, 'gasto_medio_reais': 70, 'dias_desde_ultima_compra': 55},
        {'cliente': 'C14', 'compras_mes': 1, 'gasto_medio_reais': 25, 'dias_desde_ultima_compra': 140},
        {'cliente': 'C15', 'compras_mes': 5, 'gasto_medio_reais': 130, 'dias_desde_ultima_compra': 30},
    ]

    return pd.DataFrame(dados_clientes)

In [3]:
def validar_base_clientes(base_clientes: pd.DataFrame) -> dict:
    colunas_entrada = ['compras_mes', 'gasto_medio_reais', 'dias_desde_ultima_compra']

    if base_clientes.empty:
        raise ValueError('A base de clientes não pode estar vazia.')

    valores_nulos = base_clientes.isna().sum().to_dict()
    if any(quantidade > 0 for quantidade in valores_nulos.values()):
        raise ValueError(f'Foram encontrados valores nulos: {valores_nulos}')

    for coluna in colunas_entrada:
        if not pd.api.types.is_numeric_dtype(base_clientes[coluna]):
            raise TypeError(f'A coluna {coluna} precisa ser numérica.')

        if (base_clientes[coluna] < 0).any():
            raise ValueError(f'A coluna {coluna} possui valor negativo.')

    outliers_iqr = {}
    for coluna in colunas_entrada:
        primeiro_quartil = base_clientes[coluna].quantile(0.25)
        terceiro_quartil = base_clientes[coluna].quantile(0.75)
        intervalo_iqr = terceiro_quartil - primeiro_quartil
        limite_inferior = primeiro_quartil - 1.5 * intervalo_iqr
        limite_superior = terceiro_quartil + 1.5 * intervalo_iqr
        outliers_iqr[coluna] = int(((base_clientes[coluna] < limite_inferior) | (base_clientes[coluna] > limite_superior)).sum())

    return {
        'quantidade_linhas': len(base_clientes),
        'valores_nulos': valores_nulos,
        'outliers_iqr': outliers_iqr,
    }

In [4]:
base_clientes = criar_base_clientes()
resumo_validacao_clientes = validar_base_clientes(base_clientes)

display(base_clientes)
resumo_validacao_clientes

,cliente,compras_mes,gasto_medio_reais,dias_desde_ultima_compra
0,C01,2,35,90
1,C02,1,20,120
2,C03,3,45,80
3,C04,8,180,20
4,C05,10,220,14
5,C06,7,150,25
6,C07,4,80,45
7,C08,5,95,38
8,C09,6,110,35
9,C10,12,260,7


{'quantidade_linhas': 15,
 'valores_nulos': {'cliente': 0,
  'compras_mes': 0,
  'gasto_medio_reais': 0,
  'dias_desde_ultima_compra': 0},
 'outliers_iqr': {'compras_mes': 0,
  'gasto_medio_reais': 0,
  'dias_desde_ultima_compra': 0}}

In [5]:
def interpretar_segmento(centroide_segmento: pd.Series) -> str:
    compras_mes = centroide_segmento['compras_mes']
    gasto_medio = centroide_segmento['gasto_medio_reais']
    dias_sem_comprar = centroide_segmento['dias_desde_ultima_compra']

    if compras_mes >= 8 and gasto_medio >= 180 and dias_sem_comprar <= 25:
        return 'alto valor e alta recorrência'

    if dias_sem_comprar >= 70:
        return 'risco de abandono'

    return 'recorrência moderada'

In [6]:
def segmentar_clientes(base_clientes: pd.DataFrame) -> dict:
    colunas_entrada = ['compras_mes', 'gasto_medio_reais', 'dias_desde_ultima_compra']

    normalizador_clientes = MinMaxScaler()
    dados_normalizados = normalizador_clientes.fit_transform(base_clientes[colunas_entrada])

    modelo_segmentacao = KMeans(n_clusters=3, random_state=42, n_init=20)
    grupos_clientes = modelo_segmentacao.fit_predict(dados_normalizados)

    redutor_visualizacao = PCA(n_components=2)
    coordenadas_visualizacao = redutor_visualizacao.fit_transform(dados_normalizados)

    resultado_segmentacao = base_clientes.copy()
    resultado_segmentacao['segmento'] = grupos_clientes + 1
    resultado_segmentacao['pca_1'] = coordenadas_visualizacao[:, 0]
    resultado_segmentacao['pca_2'] = coordenadas_visualizacao[:, 1]

    centroides_segmentos = pd.DataFrame(
        normalizador_clientes.inverse_transform(modelo_segmentacao.cluster_centers_),
        columns=colunas_entrada,
    )
    centroides_segmentos['segmento'] = range(1, len(centroides_segmentos) + 1)
    centroides_segmentos['perfil'] = centroides_segmentos.apply(interpretar_segmento, axis=1)

    resultado_segmentacao = resultado_segmentacao.merge(
        centroides_segmentos[['segmento', 'perfil']],
        on='segmento',
        how='left',
    )

    return {
        'resultado_segmentacao': resultado_segmentacao,
        'centroides_segmentos': centroides_segmentos,
        'silhueta_segmentacao': silhouette_score(dados_normalizados, grupos_clientes),
        'variancia_pca': redutor_visualizacao.explained_variance_ratio_,
    }


resultado_clientes = segmentar_clientes(base_clientes)

print(f'Coeficiente de silhueta: {resultado_clientes["silhueta_segmentacao"]:.2f}')
display(resultado_clientes['resultado_segmentacao'].sort_values(['segmento', 'cliente']))
display(resultado_clientes['centroides_segmentos'].round(2))

Coeficiente de silhueta: 0.50


,cliente,compras_mes,gasto_medio_reais,dias_desde_ultima_compra,segmento,pca_1,pca_2,perfil
5,C06,7,150,25,1,0.295206,-0.047632,recorrência moderada
6,C07,4,80,45,1,-0.121376,-0.139906,recorrência moderada
7,C08,5,95,38,1,-0.002087,-0.125751,recorrência moderada
8,C09,6,110,35,1,0.100776,-0.086420,recorrência moderada
12,C13,4,70,55,1,-0.187572,-0.094371,recorrência moderada
14,C15,5,130,30,1,0.118716,-0.115184,recorrência moderada
3,C04,8,180,20,2,0.443974,0.005220,alto valor e alta recorrência
4,C05,10,220,14,2,0.674826,0.101289,alto valor e alta recorrência
9,C10,12,260,7,2,0.909786,0.191064,alto valor e alta recorrência
10,C11,9,210,12,2,0.605062,0.039190,alto valor e alta recorrência


,compras_mes,gasto_medio_reais,dias_desde_ultima_compra,segmento,perfil
0,5.17,105.83,38.00,1,recorrência moderada
1,9.75,217.50,13.25,2,alto valor e alta recorrência
2,1.80,37.00,99.00,3,risco de abandono


In [7]:
figura_segmentos = px.scatter(
    resultado_clientes['resultado_segmentacao'],
    x='pca_1',
    y='pca_2',
    color='perfil',
    symbol='segmento',
    size='gasto_medio_reais',
    hover_name='cliente',
    hover_data=['compras_mes', 'gasto_medio_reais', 'dias_desde_ultima_compra'],
    title='Segmentação de clientes visualizada com PCA',
    color_discrete_sequence=['#0ca678', '#f08c00', '#4263eb'],
)
figura_segmentos.update_layout(template='plotly_white')
figura_segmentos.show()

In [8]:
distribuicao_segmentos = (
    resultado_clientes['resultado_segmentacao']
    .groupby(['segmento', 'perfil'])
    .size()
    .reset_index(name='quantidade_clientes')
    .sort_values('segmento')
)

figura_distribuicao = px.bar(
    distribuicao_segmentos,
    x='perfil',
    y='quantidade_clientes',
    color='perfil',
    text='quantidade_clientes',
    title='Quantidade de clientes por segmento',
    color_discrete_sequence=['#0ca678', '#f08c00', '#4263eb'],
)
figura_distribuicao.update_traces(textposition='outside')
figura_distribuicao.update_layout(template='plotly_white', showlegend=False)
figura_distribuicao.show()

In [9]:
centroides_para_grafico = resultado_clientes['centroides_segmentos'].copy()
centroides_para_grafico['nome_segmento'] = centroides_para_grafico['segmento'].astype(str) + ' - ' + centroides_para_grafico['perfil']

figura_centroides = px.bar(
    centroides_para_grafico,
    x='nome_segmento',
    y=['compras_mes', 'gasto_medio_reais', 'dias_desde_ultima_compra'],
    barmode='group',
    title='Perfil médio de cada segmento',
)
figura_centroides.update_layout(template='plotly_white', xaxis_title='Segmento', yaxis_title='Valor médio')
figura_centroides.show()

## Como explicar em pouco tempo

Este é um caso não supervisionado porque o modelo não recebeu a resposta correta antes. Ele só recebeu os dados dos clientes.

O K-Means agrupa clientes parecidos. O PCA não é o agrupamento em si, ele só ajuda a visualizar os grupos em duas dimensões.

Impacto prático: uma empresa poderia usar esses grupos para recuperar clientes em risco, criar campanhas para clientes de alto valor e gastar melhor o orçamento de marketing.